# A2 · Regression tree and feature importance from scratch

**Type**: Implement from scratch · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Implement a regression tree (squared error reduction, prediction = mean of the leaf) and compute the **impurity importance**: the sum, for each feature, of the impurity reductions of its splits weighted by the fraction of examples. Check that the importances match
`feature_importances_` of scikit-learn.

## Dataset

`load_diabetes` (442 patients, 10 already standardized features, target = disease progression), 70/30 split: `X_tr, y_tr, X_te, y_te`.

## Educational purpose

Understand what `feature_importances_` really is (an accounting of impurity reductions, not a causal measure) and how regression reuses the same scheme as classification with another impurity (the variance).

## Task

1. `best_split_reg(X, y, min_leaf)` → `(gain, feature, threshold)`: as in A1, but the impurity of a node is the **variance** of the target, computed with the cumulative sums of `y` and `y²`. The gain is `var(parent) − (n_L var_L + n_R var_R) / n`.
2. `build_reg(X, y, max_depth, min_leaf, N, importances, depth=0)`: recursive tree; a leaf holds the **mean** of `y`. Every time you choose a split on feature `j` add to `importances[j]` the value `gain * len(y) / N` (`N` = examples of the whole training set).
3. `predict_reg(tree, X)`. Build the tree with `max_depth=4, min_leaf=5` and save `norm_importances` (the importances divided by their sum).
4. Compare with `DecisionTreeRegressor(max_depth=4, min_samples_leaf=5, random_state=0)`: `same_importances` (`np.allclose`), `train_agreement` and `test_agreement` (fraction of predictions equal within 1e-9). Also save `top_feature` (name of the feature with the largest importance) and `top_two_share` (sum of the two largest importances).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

data = load_diabetes()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, random_state=0)
sk = DecisionTreeRegressor(max_depth=4, min_samples_leaf=5, random_state=0).fit(X_tr, y_tr)

## Your work

In [ ]:
def best_split_reg(X, y, min_leaf):
    ...

In [ ]:
def build_reg(X, y, max_depth, min_leaf, N, importances, depth=0):
    ...


def predict_reg(tree, X):
    ...

In [ ]:
importances = np.zeros(X_tr.shape[1])
tree = build_reg(X_tr, y_tr, 4, 5, len(y_tr), importances)
norm_importances = None
same_importances = None
train_agreement = None
test_agreement = None
top_feature = None
top_two_share = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Variance from cumulative sums: `cs = np.cumsum(ys); cs2 = np.cumsum(ys ** 2)`; to the left of position `i` (`nl = i + 1`): `cs2[i] / nl - (cs[i] / nl) ** 2`; to the right: `((cs2[-1] - cs2[i]) / nr) - ((cs[-1] - cs[i]) / nr) ** 2`. In the recursion: `importances[j] += gain * len(y) / N`; stop if `np.ptp(y) == 0` or `len(y) < 2 * min_leaf`.'):
    g, f, s = best_split_reg(X_tr, y_tr, 5)
    check(f == sk.tree_.feature[0] and abs(s - sk.tree_.threshold[0]) < 1e-6 and g > 0, "the split at the root matches the one of scikit-learn", "variance with the cumulative sums of y and y^2")
    check(norm_importances is not None and close(norm_importances.sum(), 1.0) and same_importances is True, "normalized importances matching scikit-learn", "sum(gain * n_node / N) per feature, then divide by the sum")
    check(train_agreement == 1.0 and test_agreement >= 0.98, "equal predictions on the training set (100%) and on at least 98% of the test", "mean of the leaf; scikit-learn uses float32 for the thresholds")
    check(top_feature == data.feature_names[int(np.argmax(sk.feature_importances_))], "the most important feature matches the one of scikit-learn", "argmax of the importances")
    check(0.7 < top_two_share < 0.9 and (norm_importances[[3, 5]] < norm_importances[8]).all(), "the top two features are worth between 70% and 90% of the importance", "sum of the two largest importances")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_regression_tree_and_importances_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.